In [2]:
from nltk.corpus import movie_reviews

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import re

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 사용자 정의 토크나이저
class SimpleTokenizer:
    def __init__(self, num_words=10000, oov_tokens='UNK'):
        self.num_words = num_words
        self.oov_tokens = oov_tokens
        self.word_index = {}
        self.index_word = {}
    def _clean_text(self, text):
        return re.findall(r'\w+', text.lower())
    def fit_on_texts(self, texts):   
        '''빈도순으로 상위 단어 추출 토큰을 숫자로 변경 공백을 기준으로 토큰분류'''
        word_counts = Counter()
        for text in texts:
            word_counts.update(self._clean_text(text))
        # 빈도순으로 num_words 단어 추출
        most_common = word_counts.most_common(self.num_words-2) # pad unk 특수토큰 자리 남겨두기
        # 0 : paddding, 1 : oov (out of vocaburay)
        self.word_index = {self.oov_tokens : 1}
        for i, (word,_) in enumerate(most_common):
            self.word_index[word] = i + 2
        self.index_word = {idx : w for w, idx in self.word_index.items()}
    def texts_to_sequence(self, texts):
        sequence = []
        for text in texts:
            seq = []
            for word in self._clean_text(text):
                seq.append(self.word_index.get(word,1))
            sequence.append(seq)
        return sequence

def pad_sequence(sequence, maxlen, padding='pre', truncating='pre'):
    features = np.zeros((len(sequence), maxlen), dtype=int)
    for i, seq in enumerate(sequence):
        if len(seq) > maxlen:
            if truncating == 'pre':
                features[i, :] = seq[-maxlen:]
            else:
                features[i, :] = seq[:maxlen]
        else:
            if padding == 'pre':
                features[i, -len(seq):] = seq
            else:
                features[i, :len(seq)] = seq
    return features


In [5]:
# 리뷰데이터로 적용해서 오류 없는지 확인 및 수정
reviews = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]

In [6]:
# RNN 모델
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embeding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embeding_dim)
        self.rnn = nn.RNN(embeding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self, x):
        x = self.embedding(x)
        _, hn = self.rnn(x)  # output, hn
        # output : 모든시점(time step)의 숨겨진 상태 -> 각 단어(시점)를 거칠때 마다 계산된 모든 hidden state를 모아놓음
        # 시점 1 ~ N 
        # 모델은 각 단어마다 결과를 내야하는 개체명 인식 (NER)
        # hn : 마지막 시점의 상태 -> 전체 문장을 다 읽고 최종적으로 요약한 정보 (문서 분류)
        return self.fc(hn.squeeze(0))

In [7]:
# 테스트
texts = reviews[0:2]
sim = SimpleTokenizer()
sim.fit_on_texts(texts) # 문자 -> 숫자
requens = sim.texts_to_sequence(texts) # 길이를 맞춤
features = pad_sequence(requens,500)
features = torch.LongTensor(features)
print(features.shape)
features = nn.Embedding(500,32)(features)
outputs, hn = nn.RNN(32,64,batch_first=True)(features)
outputs.shape, hn.shape

torch.Size([2, 500])


(torch.Size([2, 500, 64]), torch.Size([1, 2, 64]))

In [8]:
# 데이터를 가져오기 
# x, y 분할
# 토크나이저 + pad_sequence  -->  문자를 숫자로 변환

# train_test_split
# TorchTensor로 변환
# TensorDataset -> DataLoader

# 모델 생성
# 옵티마이저 정의
# 손실함수 정의

In [9]:
fileids = movie_reviews.fileids()
reviews = [movie_reviews.raw(fileid) for fileid in fileids]
categories = [movie_reviews.categories(fileid)[0] for fileid in fileids]

max_words = 10000
maxlen = 500
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = SimpleTokenizer(num_words=max_words)
tokenizer.fit_on_texts(reviews)
X = tokenizer.texts_to_sequence(reviews)
X = pad_sequence(X, maxlen=maxlen)

label_dict = {'pos': 1, 'neg': 0}
y = np.array([label_dict[c] for c in categories])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

X_train_t = torch.LongTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.LongTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print('Preprocessing complete. X_train shape:', X_train.shape)

Preprocessing complete. X_train shape: (1600, 500)


In [10]:
# def train_model(model, loader, optimizer, criterion, epochs=3):
#     for epoch in range(epochs):
#         model.train()
#         total_loss = 0
#         correct = 0
#         for X_batch, y_batch in loader:
#             X_batch, y_batch = X_batch.to(device), y_batch.to(device)
#             optimizer.zero_grad()
#             outputs = model(X_batch).squeeze()
#             loss = criterion(outputs, y_batch)
#             loss.backward()
#             optimizer.step()
#             total_loss += loss.item()
#             correct += ((outputs > 0.5).float() == y_batch).sum().item()
#         print(f'Epoch {epoch+1}: Loss {total_loss/len(loader):.4f}, Acc {correct/len(loader.dataset):.4f}')

# model_dense = RNNModel(max_words, 32, maxlen).to(device)
# criterion = nn.BCEWithLogitsLoss()
# optimizer_dense = optim.RMSprop(model_dense.parameters(), lr=0.001)
# train_model(model_dense, train_loader, optimizer_dense, criterion)

In [11]:
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(BiLSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (hn, _) = self.lstm(x)
        # 마지막 타임스텝의 정방향/역방향 은닉 상태 결합
        x = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)
        return self.fc(x)

model_bilstm = BiLSTMModel(max_words, 64, 64).to(device)
optimizer_bilstm = optim.Adam(model_bilstm.parameters(), lr=1e-4)
train_model(model_bilstm, train_loader, optimizer_bilstm, criterion, epochs=8)

NameError: name 'train_model' is not defined